# Uplift policy

This notebook extends [Uplift](uplift.ipynb) using the same `generate_obs_hte_26_rich` dataset, client characteristics, continuous LTV-style outcome, and default IRM learners.

It adds rule discovery with `UpliftPolicyTree`, independent evaluation of the frozen rules, and export of target / do-not-target client sets. The objective is positive expected uplift, with no treatment cost or capacity limit.


In [1]:
from causalis.scenarios.unconfoundedness.dgp import generate_obs_hte_26_rich

data = generate_obs_hte_26_rich(
    n=100_000, seed=42, return_causal_data=False, include_oracle=True,
)
data.head()


,user_id,y,d,tenure_months,avg_sessions_week,spend_last_month,age_years,income_monthly,prior_purchases_12m,support_tickets_90d,premium_user,mobile_user,urban_resident,referred_user,m,m_obs,tau_link,g0,g1,cate
0,1,0.000000,0.0,28.814654,1.0,77.936767,50.234101,1926.698301,1.0,2.0,1.0,1.0,1.0,0.0,0.045453,0.045453,0.089095,8.137981,9.142395,1.004414
1,2,80.099611,1.0,25.913345,3.0,53.777740,28.115859,5104.271509,3.0,0.0,1.0,1.0,0.0,1.0,0.041514,0.041514,0.246679,60.459257,78.817307,18.358049
2,3,6.400482,1.0,24.969929,10.0,134.764322,22.907062,5267.938255,8.0,3.0,0.0,1.0,1.0,0.0,0.052593,0.052593,0.162968,7.712855,9.138577,1.425723
3,4,2.788238,0.0,40.655089,5.0,59.517074,31.970490,6597.327018,3.0,2.0,1.0,1.0,1.0,0.0,0.036221,0.036221,0.188755,25.386510,31.159932,5.773422
4,5,0.000000,0.0,18.560899,3.0,74.370930,39.237248,4930.009628,5.0,1.0,1.0,1.0,0.0,0.0,0.036343,0.036343,0.174757,15.359250,18.600227,3.240977


In [2]:
print(f"Ground truth ATE is {data['cate'].mean()}")
print(f"Ground truth ATTE is {data.loc[data['d'] == 1, 'cate'].mean()}")


Ground truth ATE is 19.40958652966079
Ground truth ATTE is 10.914991423363862


## Separate training, evaluation, and new clients

The original uplift notebook samples a few fitted-data rows to illustrate scoring. Policy evaluation needs clients excluded from rule discovery, so here we split the generated population **before fitting any models**: 60% training, 30% evaluation, and 10% new clients.

Keep the evaluation split untouched while selecting features, tree depth, or other model settings. Oracle columns such as `cate`, `g0`, `g1`, and `propensity` are synthetic ground truth for demonstration only; they are excluded from both IRMs and from policy features.


In [3]:
import pandas as pd

shuffled = data.sample(frac=1, random_state=42).reset_index(drop=True)
train_end = int(0.6 * len(shuffled))
eval_end = int(0.9 * len(shuffled))
train_data = shuffled.iloc[:train_end].copy()
eval_data = shuffled.iloc[train_end:eval_end].copy()
new_data = shuffled.iloc[eval_end:].copy()

assert set(train_data.user_id).isdisjoint(eval_data.user_id)
assert set(train_data.user_id).isdisjoint(new_data.user_id)
assert set(eval_data.user_id).isdisjoint(new_data.user_id)

pd.DataFrame({
    "split": ["training", "evaluation", "new clients"],
    "n_clients": [len(train_data), len(eval_data), len(new_data)],
})


,split,n_clients
0,training,60000
1,evaluation,30000
2,new clients,10000


In [4]:
from causalis.data_contracts import CausalData

features = [
    'tenure_months',
    'avg_sessions_week',
    'spend_last_month',
    'age_years',
    'income_monthly',
    'prior_purchases_12m',
    'support_tickets_90d',
    'premium_user',
    'mobile_user',
    'urban_resident',
    'referred_user',
]

causaldata = CausalData(
    df=train_data,
    treatment='d',
    user_id='user_id',
    outcome='y',
    confounders=features,
)
causaldata


CausalData(df=(60000, 14), treatment='d', outcome='y', confounders=['tenure_months', 'avg_sessions_week', 'spend_last_month', 'age_years', 'income_monthly', 'prior_purchases_12m', 'support_tickets_90d', 'premium_user', 'mobile_user', 'urban_resident', 'referred_user'], user_id='user_id')

In [5]:
from causalis.scenarios.unconfoundedness import IRM

# Same default learners as uplift.ipynb; a seed makes the example reproducible.
model = IRM(n_jobs=-1, random_state=42).fit(causaldata)


In [6]:
dml_result = model.estimate(score='ATTE')
dml_result.summary()


,value
field,
outcome,y
estimand,ATTE
model,IRM
value,"12.0381 (ci_abs: 6.0796, 17.9966)"
value_relative,"25.8091 (ci_rel: 12.2986, 39.3197)"
alpha,0.0500
p_value,0.0001
is_significant,True
n_treated,2983


## Uplift / CATE scoring

As in the original notebook, `estimate(score='ATTE')` estimates the historical average effect among treated clients. `predict_cate(...)` estimates the conditional average effect for clients with particular pre-treatment characteristics.

The first scoring call fits and caches final full-sample outcome models. Here “full sample” means the **training split only**. Neither the evaluation clients nor the new clients enter those fits.


In [7]:
# Before uplift scoring, no final scoring models are stored on the IRM object.
(
    hasattr(model, '_uplift_g0_model_'),
    hasattr(model, '_uplift_g1_model_'),
)


(False, False)

In [8]:
# All new-client features come from the unused 10% split.
# Keep oracle truth separately; scoring and assignment receive only features/IDs.
new_clients = new_data[['user_id', *features]].copy()
new_oracle = new_data.set_index('user_id')['cate'].copy()
df_new = new_clients.sample(10, random_state=42).reset_index(drop=True)
df_new.head()


,user_id,tenure_months,avg_sessions_week,spend_last_month,age_years,income_monthly,prior_purchases_12m,support_tickets_90d,premium_user,mobile_user,urban_resident,referred_user
0,20879,12.485114,4.0,41.242767,38.254866,5160.368948,9.0,1.0,1.0,1.0,1.0,0.0
1,99376,54.551780,19.0,83.748678,39.617377,7392.953377,6.0,0.0,0.0,1.0,1.0,0.0
2,76001,18.520542,5.0,44.612308,22.749298,5033.242202,1.0,1.0,1.0,1.0,1.0,1.0
3,32102,50.720611,1.0,6.533638,66.713460,4340.416838,1.0,0.0,1.0,0.0,0.0,1.0
4,63628,50.536952,3.0,56.063084,37.107080,3390.760195,1.0,1.0,0.0,0.0,0.0,0.0


In [9]:
from causalis.scenarios.uplift import predict_cate

df_scored = df_new[['user_id']].copy()
df_scored['predicted_cate'] = predict_cate(model, df_new[features])
df_scored['oracle_cate'] = df_scored['user_id'].map(new_oracle)
# Zero-cost CATE threshold baseline, matching the policy learner's objective.
df_scored['cate_recommended_treatment'] = df_scored['predicted_cate'] > 0
df_scored


,user_id,predicted_cate,oracle_cate,cate_recommended_treatment
0,20879,-7.469850,10.688660,False
1,99376,342.110234,65.413081,True
2,76001,14.131079,8.928341,True
3,32102,-28.062179,13.587860,False
4,63628,-18.323977,0.045201,False
5,27717,10.373736,6.147334,True
6,92848,14.901758,0.116092,True
7,95953,-40.368503,9.227581,False
8,2170,37.143495,25.355212,True
9,34524,3.406375,0.446356,True


In [10]:
# The final scoring models are now cached and reused.
(
    hasattr(model, '_uplift_g0_model_'),
    hasattr(model, '_uplift_g1_model_'),
)


(True, True)

## Discover targeting rules

`UpliftPolicyTree` uses the training IRM's cross-fitted outcome and propensity estimates to construct canonical doubly robust uplift signals. It does not use `predicted_cate`, oracle effects, or the cached ATTE score to train its rules.

At each split, it maximizes the sum of positive child signal totals. Each leaf recommends treatment when its training mean signal is positive. Search is deterministic and greedy: it may miss interactions requiring an initially unhelpful split.

All eleven confounders are available for rules in this example. `policy_features` can instead name a smaller subset without removing adjustment variables from IRM. Tree settings are fixed before evaluation. Training leaf means are descriptive, not independent effect estimates; `predicates` stores exact numeric thresholds alongside readable `conditions`.


In [11]:
from causalis.scenarios.uplift import UpliftPolicyTree

policy = UpliftPolicyTree(
    max_depth=2,
    min_samples_leaf=500,
    min_samples_per_arm=30,
).fit(model, policy_features=features)

rules = policy.rules()
rules


,rule_id,conditions,predicates,action,n_train,n_treated_train,n_control_train,train_mean_uplift_descriptive
0,rule_1,spend_last_month <= 45.97638034098034 AND age_...,"((spend_last_month, <=, 45.97638034098034), (a...",1,9778,780,8998,12.831225
1,rule_2,spend_last_month <= 45.97638034098034 AND age_...,"((spend_last_month, <=, 45.97638034098034), (a...",0,13197,765,12432,-31.046664
2,rule_3,spend_last_month > 45.97638034098034 AND tenur...,"((spend_last_month, >, 45.97638034098034), (te...",1,34001,1363,32638,40.349549
3,rule_4,spend_last_month > 45.97638034098034 AND tenur...,"((spend_last_month, >, 45.97638034098034), (te...",0,3024,75,2949,-59.378080


In [12]:
policy.training_diagnostics_


{'overlap_policy': 'clip',
 'overlap_threshold': 0.01,
 'n_clipped': 1473,
 'clipped_fraction': 0.02455,
 'n_obs': 60000}

## Evaluate the frozen rules on independent clients

Fit a separate IRM on evaluation clients, using the same outcome, treatment, and confounder definitions. Its cross-fitted signals estimate the value of the already learned policy.

- `policy_vs_none`: average incremental LTV per evaluation client versus treating nobody.
- `policy_vs_all`: average incremental LTV versus treating everybody.
- `all_vs_none`: the treat-everybody baseline versus treating nobody.

Intervals use paired per-client DR signals and approximate normal inference. Rule intervals use GATE's HC3 calculation and are pointwise, not individual or simultaneous intervals. Unsupported or empty rules remain in the report with unavailable subgroup estimates. No evaluation clients are dropped from the overall comparison.

Evaluation does not alter treatment decisions. If you revise the policy using these results, a new independent sample is needed for final evaluation.


In [13]:
eval_causaldata = CausalData(
    df=eval_data,
    treatment='d',
    user_id='user_id',
    outcome='y',
    confounders=features,
)
eval_model = IRM(n_jobs=-1, random_state=43).fit(eval_causaldata)

rules_before = policy.rules()
evaluation = policy.evaluate(eval_model, alpha=0.05)
pd.testing.assert_frame_equal(rules_before, policy.rules())
evaluation.summary()


,comparison,value,std_error,ci_lower,ci_upper,n_obs,treatment_fraction
0,policy_vs_none,16.126761,7.825525,0.789014,31.464507,30000,0.729367
1,policy_vs_all,-15.620530,16.306008,-47.579718,16.338658,30000,0.729367
2,all_vs_none,31.747291,18.086125,-3.700864,67.195445,30000,0.729367


In [14]:
evaluation.rules_summary()


,rule_id,conditions,predicates,action,n_eval,n_treated,n_control,status,value,std_error,ci_lower,ci_upper
0,rule_1,spend_last_month <= 45.97638034098034 AND age_...,"((spend_last_month, <=, 45.97638034098034), (a...",1,4965,372,4593,ok,27.767834,15.920624,-3.436015,58.971683
1,rule_2,spend_last_month <= 45.97638034098034 AND age_...,"((spend_last_month, <=, 45.97638034098034), (a...",0,6575,394,6181,ok,13.196460,18.664528,-23.385343,49.778263
2,rule_3,spend_last_month > 45.97638034098034 AND tenur...,"((spend_last_month, >, 45.97638034098034), (te...",1,16916,626,16290,ok,20.450197,13.068440,-5.163475,46.063868
3,rule_4,spend_last_month > 45.97638034098034 AND tenur...,"((spend_last_month, >, 45.97638034098034), (te...",0,1544,38,1506,ok,247.311639,306.830979,-354.066029,848.689308


In [20]:
report = evaluation.rules_summary()

for row in report.itertuples():
    decision = "TARGET" if row.action == 1 else "DO NOT TARGET"
    print(f"{row.rule_id}: {decision}\n  {row.conditions}\n")

report[
    ["rule_id", "action", "n_eval", "n_treated", "n_control",
     "value", "ci_lower", "ci_upper", "status"]
].round(3)

rule_1: TARGET
  spend_last_month <= 45.97638034098034 AND age_years <= 35.00084886586084

rule_2: DO NOT TARGET
  spend_last_month <= 45.97638034098034 AND age_years > 35.00084886586084

rule_3: TARGET
  spend_last_month > 45.97638034098034 AND tenure_months <= 58.841930970952646

rule_4: DO NOT TARGET
  spend_last_month > 45.97638034098034 AND tenure_months > 58.841930970952646



,rule_id,action,n_eval,n_treated,n_control,value,ci_lower,ci_upper,status
0,rule_1,1,4965,372,4593,27.768,-3.436,58.972,ok
1,rule_2,0,6575,394,6181,13.196,-23.385,49.778,ok
2,rule_3,1,16916,626,16290,20.450,-5.163,46.064,ok
3,rule_4,0,1544,38,1506,247.312,-354.066,848.689,ok


In [15]:
# Clipping can bias estimates; report its settings and counts for both fits.
evaluation.diagnostics


{'training': {'overlap_policy': 'clip',
  'overlap_threshold': 0.01,
  'n_clipped': 1473,
  'clipped_fraction': 0.02455,
  'n_obs': 60000},
 'evaluation': {'overlap_policy': 'clip',
  'overlap_threshold': 0.01,
  'n_clipped': 1500,
  'clipped_fraction': 0.05,
  'n_obs': 30000},
 'inference': 'approximate_normal',
 'rule_intervals': 'pointwise_HC3'}

### Synthetic ground-truth check

Because this is the same synthetic scenario as `uplift.ipynb`, we can compare the evaluation estimates with the policy's oracle gain. Oracle values below describe this evaluation sample; they are not inputs to learning, and should not be used to revise the policy on this held-out sample.


In [16]:
eval_assignments = policy.assign(eval_data[['user_id', *features]], user_id='user_id')
eval_oracle = eval_assignments['user_id'].map(eval_data.set_index('user_id')['cate'])
action = eval_assignments['action']
oracle_gains = pd.Series({
    'policy_vs_none': (action * eval_oracle).mean(),
    'policy_vs_all': ((action - 1) * eval_oracle).mean(),
    'all_vs_none': eval_oracle.mean(),
}, name='oracle_gain_on_evaluation_sample')
evaluation.summary().set_index('comparison').join(oracle_gains)


,value,std_error,ci_lower,ci_upper,n_obs,treatment_fraction,oracle_gain_on_evaluation_sample
comparison,,,,,,,
policy_vs_none,16.126761,7.825525,0.789014,31.464507,30000,0.729367,15.043580
policy_vs_all,-15.620530,16.306008,-47.579718,16.338658,30000,0.729367,-4.542714
all_vs_none,31.747291,18.086125,-3.700864,67.195445,30000,0.729367,19.586295


## Assign new clients

Apply the frozen rules to the unused new-client split. Only IDs and pre-treatment policy features are required. Action `1` means target; `0` means do not target.

The table below compares tree decisions with the earlier zero-cost CATE threshold for the same ten clients. These can disagree: a shallow policy applies a shared action within each interpretable rule, whereas the CATE scorer supplies a separate model prediction for each feature vector.


In [17]:
assignments = policy.assign(new_clients, user_id='user_id')
assert assignments['user_id'].tolist() == new_clients['user_id'].tolist()

df_scored.merge(
    assignments.rename(columns={'action': 'policy_action'}),
    on='user_id', how='left', validate='one_to_one',
).sort_values('predicted_cate', ascending=False)


,user_id,predicted_cate,oracle_cate,cate_recommended_treatment,rule_id,policy_action
1,99376,342.110234,65.413081,True,rule_3,1
8,2170,37.143495,25.355212,True,rule_1,1
6,92848,14.901758,0.116092,True,rule_2,0
2,76001,14.131079,8.928341,True,rule_1,1
5,27717,10.373736,6.147334,True,rule_3,1
9,34524,3.406375,0.446356,True,rule_1,1
0,20879,-7.469850,10.688660,False,rule_2,0
4,63628,-18.323977,0.045201,False,rule_3,1
3,32102,-28.062179,13.587860,False,rule_2,0
7,95953,-40.368503,9.227581,False,rule_2,0


In [18]:
target_clients = assignments.loc[assignments['action'] == 1, ['user_id']]
do_not_target_clients = assignments.loc[assignments['action'] == 0, ['user_id']]

assert len(target_clients) + len(do_not_target_clients) == len(new_clients)
assert set(target_clients.user_id).isdisjoint(do_not_target_clients.user_id)
assert set(target_clients.user_id) | set(do_not_target_clients.user_id) == set(new_clients.user_id)

pd.DataFrame({
    'decision': ['target', 'do not target'],
    'n_clients': [len(target_clients), len(do_not_target_clients)],
})


,decision,n_clients
0,target,7319
1,do not target,2681


## Export client sets and rules

Write CSV client lists and the full assignment table to `uplift_policy_exports` relative to the notebook working directory. JSON preserves the structured numeric rule thresholds.


In [19]:
import json
from pathlib import Path

export_dir = Path('uplift_policy_exports')
export_dir.mkdir(exist_ok=True)
target_clients.to_csv(export_dir / 'target_clients.csv', index=False)
do_not_target_clients.to_csv(export_dir / 'do_not_target_clients.csv', index=False)
assignments.to_csv(export_dir / 'assignments.csv', index=False)
(export_dir / 'rules.json').write_text(
    json.dumps(policy.rules().to_dict(orient='records'), indent=2), encoding='utf-8',
)
print(f'Exports: {export_dir.resolve()}')


Exports: /Users/ioann/PycharmProjects/Ckit/notebooks/scenarios/uplift_policy_exports


Interpretation: CATE scores and rule-level effects describe average effects among clients with similar characteristics; they do not reveal a particular client's true individual treatment effect. A no-treatment decision does not establish that an individual cannot benefit.

Causal interpretation requires unconfoundedness, adequate overlap, a well-defined intervention, and independent clients without relevant spillovers. Disjoint IDs alone cannot verify these assumptions. The rich generator has relatively uncommon treatment, so inspect arm counts, clipping, and evaluation uncertainty before interpreting learned rules.
